# Этап 5. Оценка и сравнение рекомендательных моделей

## Цель работы

Провести количественную оценку качества реализованных рекомендательных алгоритмов
и сравнить их результаты на общей тестовой выборке.

В исследовании рассматриваются:

1. Popularity Baseline
2. Content-Based Filtering
3. Item-Based Collaborative Filtering
4. SVD

Для сравнения используем метрики Precision@K, Recall@K и nDCG@K.

Особое внимание уделим единому протоколу оценки, исключению утечки данных
и корректной обработке пользователей, для которых модель не может сформировать рекомендации.

## 5.1. Подготовка протокола оценки

Для всех моделей используются одинаковые обучающая и тестовая выборки, сформированные на этапе 2.

Релевантными считаются книги, которым пользователь поставил оценку не ниже 4 в тестовой выборке.

Рекомендации формируются только на основе информации, доступной в обучающей выборке.
Книги, оцененные пользователем в train, исключаются из выдачи.

Основные параметры эксперимента:

- количество рекомендаций: K = 5
- порог релевантности: rating >= 4
- тестовые оценки используются только для расчета метрик
- все модели оцениваются на одинаковом наборе пользователей
- отсутствие рекомендаций учитывается при усреднении метрик

Для первоначального сравнения отберем воспроизводимую подвыборку пользователей с релевантными тестовыми взаимодействиями.

Отдельно оценим покрытие пользователей рекомендациями, поскольку некоторые алгоритмы могут возвращать менее K книг.

Гиперпараметры моделей не подбираются на итоговой тестовой выборке.

In [1]:
# === Импорты ===

from time import perf_counter

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sklearn.feature_extraction.text import TfidfVectorizer

from homework_tasks.recommendation_systems.homework_goodbooks.src.config import (
    RANDOM_STATE,
    RELEVANCE_THRESHOLD,
    TEST_RATINGS_PATH,
    TOP_K,
    TRAIN_RATINGS_PATH,
    MAX_TAGS_PER_BOOK,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.evaluation import (
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
    evaluate_recommender,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.data_loader import (
    load_book_tags,
    load_books,
    load_tags,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.data_preparation import (
    prepare_book_content,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.recommenders import (
    build_popularity_ranking,
    get_content_recommendations,
    get_item_recommendations,
    get_popular_recommendations,
)
from homework_tasks.recommendation_systems.homework_goodbooks.src.svd_model import (
    get_svd_recommendations,
    train_svd_model,
)

In [2]:
# === Загрузка обучающей и тестовой выборок ===

train_ratings = pd.read_csv(TRAIN_RATINGS_PATH)
test_ratings = pd.read_csv(TEST_RATINGS_PATH)

print(f"Train: {train_ratings.shape}")
print(f"Test: {test_ratings.shape}")
print(f"Top-K: {TOP_K}")
print(f"Порог релевантности: {RELEVANCE_THRESHOLD}")

Train: (788503, 3)
Test: (190975, 3)
Top-K: 5
Порог релевантности: 4


In [3]:
# === Определение релевантных книг в тестовой выборке ===

relevant_test_ratings = test_ratings.loc[
    test_ratings["rating"] >= RELEVANCE_THRESHOLD
    ].copy()

relevant_books_per_user = (
    relevant_test_ratings.groupby("user_id")["book_id"]
    .nunique()
)

print(f"Всего тестовых взаимодействий: {len(test_ratings)}")
print(f"Релевантных взаимодействий: {len(relevant_test_ratings)}")
print(f"Пользователей в test: {test_ratings['user_id'].nunique()}")
print(f"Пользователей с релевантными книгами: {len(relevant_books_per_user)}")

display(relevant_books_per_user.describe())

Всего тестовых взаимодействий: 190975
Релевантных взаимодействий: 126704
Пользователей в test: 53382
Пользователей с релевантными книгами: 42654


count    42654.000000
mean         2.970507
std          3.860475
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max         38.000000
Name: book_id, dtype: float64

In [4]:
# === Проверка отсутствия пересечений train и test ===

train_pairs = pd.MultiIndex.from_frame(
    train_ratings[["user_id", "book_id"]]
)

test_pairs = pd.MultiIndex.from_frame(
    test_ratings[["user_id", "book_id"]]
)

shared_pairs_count = len(train_pairs.intersection(test_pairs))

print(f"Общих пар пользователь–книга: {shared_pairs_count}")

assert shared_pairs_count == 0
assert relevant_test_ratings["rating"].ge(RELEVANCE_THRESHOLD).all()
assert TOP_K > 0

Общих пар пользователь–книга: 0


### Выводы по подготовке протокола оценки

По результатам анализа обучающей и тестовой выборок установлено:

1. Обучающая выборка содержит 788 503 оценки, тестовая - 190 975 оценок.

2. При пороге релевантности 4 в тестовой выборке выявлено 126 704 релевантных взаимодействия.

3. Из 53 382 пользователей тестовой выборки 42 654 имеют хотя бы одну релевантную книгу.

4. Среднее число релевантных книг составляет 2,97, медиана - 1.
Следовательно, у значительной части пользователей имеется очень небольшое количество
известных положительных взаимодействий.

5. Пересечений пар пользователь–книга между обучающей и тестовой выборками не обнаружено.

6. Для всех алгоритмов будут использоваться одинаковые пользователи,
общий порог релевантности и одинаковая длина рекомендательного списка K = 5.

**Ограничение оценки:**

Отсутствие книги среди тестовых оценок пользователя не означает, что она ему не понравилась бы.
Поэтому метрики будут отражать попадание рекомендаций в наблюдаемые релевантные взаимодействия.

**Итог:**

Подготовлен единый протокол offline-оценки рекомендательных алгоритмов.

Для первого эксперимента используем фиксированную воспроизводимую подвыборку пользователей
с релевантными тестовыми взаимодействиями.

In [5]:
# === Формирование фиксированной выборки пользователей для оценки ===

evaluation_users_count = 200

random_generator = np.random.default_rng(RANDOM_STATE)

eligible_user_ids = np.sort(
    relevant_books_per_user.index.to_numpy()
)

evaluation_user_ids = random_generator.choice(
    eligible_user_ids,
    size=evaluation_users_count,
    replace=False,
)

evaluation_user_ids = np.sort(evaluation_user_ids)

print(f"Количество пользователей для оценки: {len(evaluation_user_ids)}")
print(f"Уникальных пользователей: {len(np.unique(evaluation_user_ids))}")
print(f"Первые 10 ID: {evaluation_user_ids[:10]}")

assert len(evaluation_user_ids) == evaluation_users_count
assert len(np.unique(evaluation_user_ids)) == evaluation_users_count

Количество пользователей для оценки: 200
Уникальных пользователей: 200
Первые 10 ID: [ 412 1276 1706 1915 2057 2391 3215 3509 3738 4189]


## 5.2. Реализация метрик качества рекомендаций

Для количественного сравнения моделей реализуем три метрики:

- **Precision@K** -> доля релевантных рекомендаций среди K позиций списка
- **Recall@K** -> доля найденных релевантных книг среди всех известных релевантных книг пользователя
- **nDCG@K** -> нормализованная метрика ранжирования, учитывающая позиции релевантных книг

Для вычислений используем бинарную релевантность:
книга считается релевантной, если ее тестовая оценка не ниже 4.

Метрики рассчитываются отдельно для каждого пользователя,
после чего усредняются по общей выборке.

Если алгоритм возвращает менее K рекомендаций, незаполненные позиции не учитываются как успешные попадания.
Для Precision@K знаменатель остается равным K.

При отсутствии рекомендаций все три метрики принимают значение 0.

In [6]:
# === Проверка метрик на простом примере ===

sample_recommended_ids = [10, 20, 30, 40, 50]
sample_relevant_ids = {10, 30, 60}

precision = precision_at_k(
    recommended_ids=sample_recommended_ids,
    relevant_ids=sample_relevant_ids,
    k=5,
)

recall = recall_at_k(
    recommended_ids=sample_recommended_ids,
    relevant_ids=sample_relevant_ids,
    k=5,
)

ndcg = ndcg_at_k(
    recommended_ids=sample_recommended_ids,
    relevant_ids=sample_relevant_ids,
    k=5,
)

print(f"Precision@5: {precision:.4f}")
print(f"Recall@5: {recall:.4f}")
print(f"nDCG@5: {ndcg:.4f}")

assert np.isclose(precision, 0.4)
assert np.isclose(recall, 2 / 3)
assert 0 <= ndcg <= 1

Precision@5: 0.4000
Recall@5: 0.6667
nDCG@5: 0.7039


In [7]:
# === Проверка граничных случаев метрик ===

# Нет рекомендаций
assert precision_at_k([], {1, 2}, k=5) == 0
assert recall_at_k([], {1, 2}, k=5) == 0
assert ndcg_at_k([], {1, 2}, k=5) == 0

# Все рекомендации релевантны
assert precision_at_k([1, 2, 3], {1, 2, 3}, k=3) == 1
assert recall_at_k([1, 2, 3], {1, 2, 3}, k=3) == 1
assert np.isclose(ndcg_at_k([1, 2, 3], {1, 2, 3}, k=3), 1)

# Релевантные книги отсутствуют
assert recall_at_k([1, 2], set(), k=5) == 0
assert ndcg_at_k([1, 2], set(), k=5) == 0

print("Проверки метрик успешно выполнены")

Проверки метрик успешно выполнены


### Выводы по реализации метрик

Реализованы функции расчета Precision@K, Recall@K и nDCG@K.

На контрольном примере получены следующие значения:

- Precision@5 = 0,4000
- Recall@5 = 0,6667
- nDCG@5 = 0,7039

Результаты соответствуют ожидаемым значениям.
Проверки пустых рекомендаций, отсутствия релевантных книг и полностью релевантной выдачи завершились успешно.

Для итогового сравнения выбрана воспроизводимая подвыборка
из 200 пользователей с положительными тестовыми взаимодействиями.

## 5.3. Персональные Content-Based рекомендации

Ранее была реализована функция поиска похожих книг на основе TF-IDF и косинусной близости.

Для сравнения с другими алгоритмами необходимо формировать рекомендации не по одной книге,
а по истории предпочтений пользователя.

Для этого построим текстовый профиль пользователя как среднее TF-IDF векторов книг,
оцененных им не ниже 4 в обучающей выборке.

Затем рассчитаем косинусную близость между профилем пользователя и книгами каталога.

Все книги, оцененные пользователем в train, исключим из рекомендаций независимо от величины оценки.

Если положительных оценок нет, алгоритм не формирует рекомендации.

In [8]:
# === Подготовка Content-Based модели для оценки ===

books = load_books()

books_content = prepare_book_content(
    books=books,
    book_tags=load_book_tags(),
    tags=load_tags(),
    max_tags_per_book=MAX_TAGS_PER_BOOK,
)

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=2,
    max_df=0.95,
    norm="l2",
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    books_content["content_text"]
)

print(f"Количество книг: {len(books_content)}")
print(f"Размер TF-IDF матрицы: {tfidf_matrix.shape}")

assert tfidf_matrix.shape[0] == len(books_content)
assert books_content["id"].is_unique

Количество книг: 10000
Размер TF-IDF матрицы: (10000, 5812)


In [9]:
# === Персональные Content-Based рекомендации ===

sample_user_id = 12874

content_recommendations = get_content_recommendations(
    user_id=sample_user_id,
    train_ratings=train_ratings,
    books_content=books_content,
    tfidf_matrix=tfidf_matrix,
    top_n=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

print(f"Пользователь: {sample_user_id}")
print(f"Количество рекомендаций: {len(content_recommendations)}")

display(content_recommendations)

Пользователь: 12874
Количество рекомендаций: 5


,book_id,title,authors,similarity
0,1896,"Pollyanna (Pollyanna, #1)",Eleanor H. Porter,0.714414
1,540,A Little Princess,"Frances Hodgson Burnett, Nancy Bond",0.689795
2,1497,"The Borrowers (The Borrowers, #1)","Mary Norton, Beth Krush, Joe Krush",0.686375
3,9255,Little Lord Fauntleroy,Frances Hodgson Burnett,0.678879
4,289,"Watership Down (Watership Down, #1)",Richard Adams,0.677668


In [10]:
# === Проверка персональных Content-Based рекомендаций ===

rated_book_ids = set(
    train_ratings.loc[
        train_ratings["user_id"] == sample_user_id,
        "book_id",
    ]
)

recommended_book_ids = set(content_recommendations["book_id"])

print(
    "Уже оцененных книг в рекомендациях:",
    len(rated_book_ids & recommended_book_ids),
)

assert len(content_recommendations) <= TOP_K
assert content_recommendations["book_id"].is_unique
assert not (rated_book_ids & recommended_book_ids)
assert content_recommendations["similarity"].is_monotonic_decreasing
assert content_recommendations["similarity"].between(0, 1).all()

Уже оцененных книг в рекомендациях: 0


### Выводы по персональным Content-Based рекомендациям

По результатам реализации персональных Content-Based рекомендаций установлено:

1. Для всех 10 000 книг подготовлены текстовые профили и построена TF-IDF матрица размерностью 10 000 x 5 812.

2. Реализована функция `get_content_recommendations()`,
которая формирует пользовательский профиль на основе книг,
оцененных не ниже 4 в обучающей выборке.

3. Для пользователя `12874` получено пять рекомендаций с косинусной близостью от 0,6777 до 0,7144.

4. В рекомендательный список вошли `Pollyanna`, `A Little Princess`, `The Borrowers`, `Little Lord Fauntleroy` и `Watership Down`.

5. Полученные рекомендации преимущественно относятся к детской,
классической и приключенческой литературе.

6. Ранее оцененные пользователем книги исключены, дубликатов нет,
рекомендации отсортированы по убыванию сходства.

7. В отличие от поиска похожих произведений по одной книге,
персональная модель учитывает совокупность положительных оценок пользователя.

**Итог:**

Реализован персональный Content-Based алгоритм, пригодный для сравнения с Popularity Baseline, Item-Based CF и SVD.

При построении рекомендаций используются только обучающие оценки пользователя.
При этом текстовые признаки книг получены из каталога и пользовательских тегов,
историческая доступность которых не подтверждена из-за отсутствия временных меток.

## 5.4. Подготовка моделей к общему сравнению

In [11]:
# === Подготовка моделей Popularity и Item-Based CF ===

popularity_ranking = build_popularity_ranking(
    train_ratings=train_ratings,
    min_ratings_count=20,
)

user_ids = np.sort(train_ratings["user_id"].unique())
book_ids = np.sort(train_ratings["book_id"].unique())

user_to_index = {
    user_id: index
    for index, user_id in enumerate(user_ids)
}

book_to_index = {
    book_id: index
    for index, book_id in enumerate(book_ids)
}

row_indices = train_ratings["user_id"].map(user_to_index).to_numpy()
column_indices = train_ratings["book_id"].map(book_to_index).to_numpy()
rating_values = train_ratings["rating"].to_numpy(dtype=np.float32)

user_item_matrix = csr_matrix(
    (rating_values, (row_indices, column_indices)),
    shape=(len(user_ids), len(book_ids)),
)

item_user_matrix = user_item_matrix.T.tocsr()

print(f"Книг в Popularity Baseline: {len(popularity_ranking)}")
print(f"Размер матрицы взаимодействий: {user_item_matrix.shape}")
print(f"Ненулевых элементов: {user_item_matrix.nnz}")

assert user_item_matrix.nnz == len(train_ratings)
assert item_user_matrix.shape == (
    len(book_ids),
    len(user_ids),
)

Книг в Popularity Baseline: 9997
Размер матрицы взаимодействий: (53424, 10000)
Ненулевых элементов: 788503


In [12]:
# === Обучение SVD для общего сравнения ===

svd_model = train_svd_model(
    train_ratings=train_ratings,
    n_factors=50,
    n_epochs=20,
    random_state=RANDOM_STATE,
)

print(f"Размерность факторов пользователей: {svd_model.pu.shape}")
print(f"Размерность факторов книг: {svd_model.qi.shape}")
print("SVD готова к сравнению")

Размерность факторов пользователей: (53424, 50)
Размерность факторов книг: (10000, 50)
SVD готова к сравнению


In [13]:
# === Проверка Top-5 рекомендаций всех моделей ===

sample_evaluation_user_id = int(evaluation_user_ids[0])

popular_recommendations = get_popular_recommendations(
    popularity_ranking=popularity_ranking,
    books=books,
    top_n=TOP_K,
    user_id=sample_evaluation_user_id,
    train_ratings=train_ratings,
)

content_recommendations = get_content_recommendations(
    user_id=sample_evaluation_user_id,
    train_ratings=train_ratings,
    books_content=books_content,
    tfidf_matrix=tfidf_matrix,
    top_n=TOP_K,
    relevance_threshold=RELEVANCE_THRESHOLD,
)

item_recommendations = get_item_recommendations(
    user_id=sample_evaluation_user_id,
    user_item_matrix=user_item_matrix,
    item_user_matrix=item_user_matrix,
    user_to_index=user_to_index,
    book_ids=book_ids,
    books=books,
    top_n=TOP_K,
    top_k=10,
    min_positive_neighbors=3,
)

svd_recommendations = get_svd_recommendations(
    model=svd_model,
    user_id=sample_evaluation_user_id,
    books=books,
    top_n=TOP_K,
)

model_recommendations = {
    "Popularity": popular_recommendations,
    "Content-Based": content_recommendations,
    "Item-Based CF": item_recommendations,
    "SVD": svd_recommendations,
}

print(f"Пользователь: {sample_evaluation_user_id}")

for model_name, recommendations in model_recommendations.items():
    print(f"\n{model_name}: {len(recommendations)} рекомендаций")
    display(recommendations)

Пользователь: 412

Popularity: 5 рекомендаций


,book_id,title,authors,ratings_count,average_rating
0,6920,The Indispensable Calvin and Hobbes,Bill Watterson,73,4.835616
1,7947,ESV Study Bible,"Anonymous, Lane T. Dennis, Wayne A. Grudem",67,4.820896
2,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,85,4.811765
3,8663,"Locke & Key, Vol. 6: Alpha & Omega","Joe Hill, Gabriel Rodríguez",82,4.792683
4,3275,"Harry Potter Boxed Set, Books 1-5 (Harry Potte...","J.K. Rowling, Mary GrandPré",86,4.790698



Content-Based: 5 рекомендаций


,book_id,title,authors,similarity
0,5346,Where I'm Calling From: New and Selected Stories,Raymond Carver,0.592188
1,8818,"Will You Please Be Quiet, Please?",Raymond Carver,0.578509
2,3187,No One Belongs Here More Than You,Miranda July,0.567191
3,4055,Jesus' Son,Denis Johnson,0.566495
4,6892,CivilWarLand in Bad Decline,George Saunders,0.547240



Item-Based CF: 5 рекомендаций


,book_id,title,authors,predicted_rating
0,5033,The Postman Always Rings Twice,James M. Cain,5.0
1,1334,The Tiger's Wife,Téa Obreht,5.0
2,7979,Born to Run,Bruce Springsteen,5.0
3,8124,Zone One,Colson Whitehead,5.0
4,8298,Story of a Girl,Sara Zarr,5.0



SVD: 5 рекомендаций


,book_id,title,authors,predicted_rating
0,5580,The Calvin and Hobbes Lazy Sunday Book,Bill Watterson,5.000000
1,4868,Jesus the Christ,James E. Talmage,4.935536
2,6920,The Indispensable Calvin and Hobbes,Bill Watterson,4.900470
3,5207,The Days Are Just Packed: A Calvin and Hobbes ...,Bill Watterson,4.894493
4,1788,The Calvin and Hobbes Tenth Anniversary Book,Bill Watterson,4.848251


In [14]:
# === Проверка результатов четырех моделей ===

sample_rated_book_ids = set(
    train_ratings.loc[
        train_ratings["user_id"] == sample_evaluation_user_id,
        "book_id",
    ]
)

for model_name, recommendations in model_recommendations.items():
    recommended_ids = recommendations["book_id"].tolist()

    assert len(recommended_ids) <= TOP_K
    assert len(recommended_ids) == len(set(recommended_ids))
    assert not (set(recommended_ids) & sample_rated_book_ids)
    assert recommendations["book_id"].notna().all()

    print(f"{model_name}: проверки пройдены")

print("Все модели соответствуют единому формату рекомендаций")

Popularity: проверки пройдены
Content-Based: проверки пройдены
Item-Based CF: проверки пройдены
SVD: проверки пройдены
Все модели соответствуют единому формату рекомендаций


### Выводы по подготовке моделей

Для общего сравнения подготовлены четыре рекомендательных алгоритма.

Все модели проверены на пользователе `412` и успешно сформировали Top-5 рекомендации.

Подтверждены отсутствие повторяющихся книг, корректность идентификаторов
и исключение ранее оцененных произведений.

Полученные списки отличаются по составу, поскольку модели используют разные принципы формирования рекомендаций.

## 5.5. Расчет метрик и сравнение моделей

Для количественного сравнения используем единый механизм оценки,
который рассчитывает Precision@5, Recall@5 и nDCG@5 отдельно для каждого пользователя.

Дополнительно учитываем количество сформированных рекомендаций,
чтобы оценить покрытие пользователей.

Сначала выполним проверочный расчет на 10 пользователях,
затем расширим эксперимент до фиксированной выборки из 200 пользователей.

Все модели используют ранее зафиксированные параметры.
Тестовые оценки применяются только для вычисления метрик,
но не для формирования рекомендаций.

In [15]:
# === Единый интерфейс получения рекомендаций ===

def recommend_popularity(user_id: int) -> pd.DataFrame:
    """
    Возвращает Top-K рекомендаций Popularity Baseline.
    """

    return get_popular_recommendations(
        popularity_ranking=popularity_ranking,
        books=books,
        top_n=TOP_K,
        user_id=user_id,
        train_ratings=train_ratings,
    )


def recommend_content(user_id: int) -> pd.DataFrame:
    """
    Возвращает Top-K персональных Content-Based рекомендаций.
    """

    return get_content_recommendations(
        user_id=user_id,
        train_ratings=train_ratings,
        books_content=books_content,
        tfidf_matrix=tfidf_matrix,
        top_n=TOP_K,
        relevance_threshold=RELEVANCE_THRESHOLD,
    )


def recommend_item(user_id: int) -> pd.DataFrame:
    """
    Возвращает Top-K рекомендаций Item-Based CF.
    """

    return get_item_recommendations(
        user_id=user_id,
        user_item_matrix=user_item_matrix,
        item_user_matrix=item_user_matrix,
        user_to_index=user_to_index,
        book_ids=book_ids,
        books=books,
        top_n=TOP_K,
        top_k=10,
        min_positive_neighbors=3,
    )


def recommend_svd(user_id: int) -> pd.DataFrame:
    """
    Возвращает Top-K рекомендаций SVD.
    """

    return get_svd_recommendations(
        model=svd_model,
        user_id=user_id,
        books=books,
        top_n=TOP_K,
    )


recommenders = {
    "Popularity": recommend_popularity,
    "Content-Based": recommend_content,
    "Item-Based CF": recommend_item,
    "SVD": recommend_svd,
}

print(f"Подготовлено моделей: {len(recommenders)}")

Подготовлено моделей: 4


In [16]:
# === Проверка расчета метрик для четырех моделей ===

pilot_user_ids = evaluation_user_ids[:10]

pilot_results = {}

for model_name, recommend in recommenders.items():
    model_results = evaluate_recommender(
        user_ids=pilot_user_ids,
        relevant_test_ratings=relevant_test_ratings,
        recommend=recommend,
        k=TOP_K,
    )

    pilot_results[model_name] = {
        "precision": model_results["precision"].mean(),
        "recall": model_results["recall"].mean(),
        "ndcg": model_results["ndcg"].mean(),
        "users_with_recommendations": (
            model_results["recommendations_count"] > 0
        ).mean(),
        "users_with_full_top_k": (
            model_results["recommendations_count"] == TOP_K
        ).mean(),
    }

pilot_results_df = pd.DataFrame.from_dict(
    pilot_results,
    orient="index",
)

display(pilot_results_df.round(4))

,precision,recall,ndcg,users_with_recommendations,users_with_full_top_k
Popularity,0.00,0.0000,0.0000,1.0,1.0
Content-Based,0.04,0.0667,0.0765,1.0,1.0
Item-Based CF,0.00,0.0000,0.0000,0.8,0.8
SVD,0.00,0.0000,0.0000,1.0,1.0


In [17]:
# === Проверка метрик предварительного сравнения ===

assert len(pilot_results_df) == 4
assert pilot_results_df.notna().all().all()
assert ((pilot_results_df >= 0) & (pilot_results_df <= 1)).all().all()

print("Предварительное сравнение четырех моделей выполнено")

Предварительное сравнение четырех моделей выполнено


### Результаты предварительного сравнения

Для проверки единого механизма оценки выполнен пилотный эксперимент на 10 пользователях.

Получены следующие результаты:

1. Popularity Baseline: Precision@5, Recall@5 и nDCG@5 равны 0.

2. Content-Based Filtering: Precision@5 = 0,0400, Recall@5 = 0,0667, nDCG@5 = 0,0765.

3. Item-Based Collaborative Filtering: все три метрики равны 0.
Рекомендации сформированы для 80% пользователей.

4. SVD: все три метрики равны 0.

5. Popularity, Content-Based и SVD сформировали полный Top-5 для всех пользователей пилотной выборки.

**Интерпретация:**

В предварительном эксперименте только Content-Based модель смогла рекомендовать книги,
присутствующие среди релевантных тестовых взаимодействий.

При этом выборка из 10 пользователей слишком мала для достоверного ранжирования моделей по качеству.

Результаты подтверждают работоспособность общего механизма оценки
и показывают необходимость учитывать покрытие пользователей.

**Итог:**

Пилотный эксперимент успешно завершен.
Можно переходить к сравнению на полной фиксированной выборке из 200 пользователей.

In [18]:
# === Полное сравнение моделей на 200 пользователях ===

evaluation_results = {}
evaluation_summary = []

for model_name, recommend in recommenders.items():
    start_time = perf_counter()

    model_results = evaluate_recommender(
        user_ids=evaluation_user_ids,
        relevant_test_ratings=relevant_test_ratings,
        recommend=recommend,
        k=TOP_K,
    )

    elapsed_seconds = perf_counter() - start_time

    evaluation_results[model_name] = model_results

    evaluation_summary.append(
        {
            "model": model_name,
            "precision": model_results["precision"].mean(),
            "recall": model_results["recall"].mean(),
            "ndcg": model_results["ndcg"].mean(),
            "users_with_recommendations": (
                model_results["recommendations_count"] > 0
            ).mean(),
            "users_with_full_top_k": (
                model_results["recommendations_count"] == TOP_K
            ).mean(),
            "execution_time_seconds": elapsed_seconds,
        }
    )

evaluation_summary_df = pd.DataFrame(evaluation_summary)

display(evaluation_summary_df.round(4))

,model,precision,recall,ndcg,users_with_recommendations,users_with_full_top_k,execution_time_seconds
0,Popularity,0.000,0.0000,0.0000,1.000,1.000,0.7458
1,Content-Based,0.043,0.1034,0.0861,0.955,0.955,0.7330
2,Item-Based CF,0.003,0.0105,0.0052,0.780,0.780,1.4512
3,SVD,0.001,0.0010,0.0007,1.000,1.000,0.6792


In [19]:
# === Проверка итогового сравнения ===

assert len(evaluation_summary_df) == len(recommenders)

for model_name, model_results in evaluation_results.items():
    assert len(model_results) == len(evaluation_user_ids)
    assert model_results["user_id"].is_unique
    assert set(model_results["user_id"]) == set(evaluation_user_ids)

    metric_columns = ["precision", "recall", "ndcg"]

    assert model_results[metric_columns].notna().all().all()
    assert (
        (model_results[metric_columns] >= 0)
        & (model_results[metric_columns] <= 1)
    ).all().all()

    assert model_results["recommendations_count"].between(
        0, TOP_K
    ).all()

    print(f"{model_name}: результаты проверены")

print("Итоговое сравнение четырех моделей выполнено")

Popularity: результаты проверены
Content-Based: результаты проверены
Item-Based CF: результаты проверены
SVD: результаты проверены
Итоговое сравнение четырех моделей выполнено


In [20]:
# === Дополнительный анализ результатов оценки ===

additional_results = []

for model_name, model_results in evaluation_results.items():
    additional_results.append(
        {
            "model": model_name,
            "users_count": len(model_results),
            "users_with_hits": int(
                (model_results["precision"] > 0).sum()
            ),
            "users_without_recommendations": int(
                (model_results["recommendations_count"] == 0).sum()
            ),
            "users_with_full_top_k": int(
                (model_results["recommendations_count"] == TOP_K).sum()
            ),
            "average_recommendations_count": (
                model_results["recommendations_count"].mean()
            ),
        }
    )

additional_results_df = pd.DataFrame(additional_results)

display(additional_results_df.round(4))

,model,users_count,users_with_hits,users_without_recommendations,users_with_full_top_k,average_recommendations_count
0,Popularity,200,0,0,200,5.000
1,Content-Based,200,36,9,191,4.775
2,Item-Based CF,200,3,44,156,3.900
3,SVD,200,1,0,200,5.000


### Выводы по сравнению рекомендательных моделей

По результатам оценки четырех моделей на фиксированной выборке из 200 пользователей установлено:

1. **Content-Based Filtering** показал наилучшие результаты по всем трем метрикам:
Precision@5 = 0,0430, Recall@5 = 0,1034, nDCG@5 = 0,0861.

2. Content-Based обеспечил 43 попадания в релевантные тестовые взаимодействия.
Хотя бы одно попадание получено для 36 пользователей.

3. **Item-Based Collaborative Filtering** получил Precision@5 = 0,0030, Recall@5 = 0,0105 и nDCG@5 = 0,0052.
При этом 44 пользователя не получили рекомендаций.

4. **SVD** обеспечил полный Top-5 для всех пользователей,
однако получил низкие показатели качества:
Precision@5 = 0,0010, Recall@5 = 0,0010, nDCG@5 = 0,0007.

5. **Popularity Baseline** также сформировал полный Top-5 для всех пользователей,
но не получил попаданий в релевантные тестовые взаимодействия.

6. По времени формирования рекомендаций наиболее быстрыми в данном эксперименте оказались SVD и Popularity Baseline.
Наибольшее время потребовалось Item-Based CF.

7. Покрытие пользователей и качество рекомендаций не совпадают:
алгоритм может формировать полный список,
но не находить наблюдаемые релевантные книги.

**Интерпретация:**

В рамках проведенного эксперимента Content-Based Filtering оказался наиболее результативным подходом.

Возможным объяснением является использование тематических тегов книг
и формирование профиля на основе положительных оценок пользователя.

При этом результаты не доказывают универсального превосходства Content-Based.
На них влияют разреженность данных, особенности разбиения,
состав тестовой выборки и принятый способ определения релевантности.

**Итог:**

Единый механизм оценки позволил количественно сравнить четыре рекомендательных алгоритма.

Для текущего набора данных и зафиксированных параметров Content-Based показал лучшие offline-метрики.
Однако окончательные выводы необходимо рассматривать с учетом ограничений эксперимента.

## 5.6. Ограничения эксперимента и итоговые выводы

### Ограничения offline-оценки

При интерпретации полученных результатов необходимо учитывать следующие ограничения.

**1. Неполнота наблюдаемых взаимодействий**

В тестовой выборке отсутствуют оценки большинства книг для каждого пользователя.

Поэтому отсутствие рекомендованной книги среди релевантных тестовых взаимодействий
не означает ее фактическую нерелевантность.

**2. Высокая разреженность**

Матрица взаимодействий имеет разреженность более 99,8%.
Это затрудняет определение сходства книг
и обучение персональных моделей на пользовательских оценках.

**3. Ограниченная выборка пользователей**

Эксперимент выполнен на 200 пользователях,
имеющих хотя бы одно релевантное тестовое взаимодействие.

Результаты не следует автоматически распространять
на всех пользователей датасета.

**4. Случайное разделение данных**

Train/test-разделение выполнено по пользователям
без учета времени взаимодействий.

Такой способ позволяет проверить алгоритмы на отложенных оценках,
но не воспроизводит полностью реальный сценарий
последовательного появления пользовательских событий.

**5. Текстовые признаки Content-Based**

Профили книг построены на основе названий и пользовательских тегов.

Теги получены из общего каталога и не имеют временных меток, позволяющих подтвердить,
что они были доступны на момент формирования обучающей выборки.

Поэтому нельзя полностью исключить влияние информации,
сформировавшейся после отдельных тестовых взаимодействий.

**6. Гиперпараметры и статистическая устойчивость**

Для моделей использованы предварительно заданные параметры
без полноценного подбора на отдельной валидационной выборке.

Проведен один эксперимент на фиксированной подвыборке пользователей.
Статистическая значимость различий между моделями отдельно не проверялась.

### Итоговые выводы

В рамках пятого этапа:

1. Реализованы метрики Precision@K, Recall@K и nDCG@K.
2. Подготовлен единый механизм оценки рекомендательных алгоритмов.
3. Реализованы персональные Content-Based рекомендации.
4. Выполнено сравнение четырех моделей на 200 пользователях.
5. Дополнительно исследованы покрытие пользователей и время формирования рекомендаций.
6. Выявлены ограничения offline-оценки и особенности работы моделей на разреженных данных.

**Общий вывод:**

Наилучшие показатели Precision@5, Recall@5 и nDCG@5 в проведенном эксперименте получила Content-Based модель.

Popularity Baseline и SVD обеспечили полное покрытие выбранных пользователей,
однако не показали сопоставимого качества попадания в релевантные тестовые взаимодействия.

Item-Based CF продемонстрировал чувствительность к разреженности данных
и установленному ограничению количества положительных соседей.

Полученные результаты подтверждают необходимость оценивать рекомендательные алгоритмы не только по отдельным примерам,
но и по единым метрикам, с учетом покрытия и ограничений данных.

Для дальнейшего развития системы целесообразно рассмотреть отдельную валидацию гиперпараметров,
временное разделение взаимодействий и резервные рекомендации для пользователей с недостаточной историей.